In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
loader = TextLoader("fpt_policy.txt", encoding="utf-8")
documents = loader.load()

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)

In [5]:
gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma.from_documents(texts, gemini_embeddings, collection_name="fpt_policy")

In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [7]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0.0
)

@tool
def search_fpt_policy(query: str) -> str:
    """
    Hãy sử dụng công cụ này BẮT BUỘC khi bạn cần tìm kiếm thông tin về quy định, 
    chính sách, nội quy, giờ giấc, lương thưởng hoặc bất kỳ vấn đề nhân sự nào của FPT.
    """
   
    results = retriever.invoke(query) 
    
    return "\n\n---\n\n".join([result.page_content for result in results]) #gom 3 kết quả tìm được thành 1 đoạn văn bản duy nhất, ngăn cách bằng "---"
# @tool
# def policy_decision(context: str, question: str) -> str:
#     """
#     Đưa ra quyết định dựa trên policy.
#     """
#     prompt = f"""
#     Policy:
#     {context}

#     Question:
#     {question}

#     Trả lời có/không + giải thích.
#     """
#     return llm.invoke(prompt).content
tools = [search_fpt_policy]

In [8]:
SYSTEM_PROMPT = """
Bạn là một trợ lý Nhân sự (HR) mẫn cán của tập đoàn FPT. 
Nhiệm vụ của bạn là giải đáp thắc mắc của nhân viên về các quy định nội bộ.

Quy tắc bắt buộc:
1. LUÔN LUÔN sử dụng công cụ 'search_fpt_policy' trước khi trả lời bất kỳ câu hỏi nào về quy định công ty.
2. Tuyệt đối không tự bịa thông tin. Dựa 100% vào dữ liệu tìm được.
3. Hãy trả lời ngắn gọn, lịch sự.
"""
# mmodel = init_chat_model(model="openai:gpt-4.1-mini", temperature=0.0)
agent = create_agent(
    model=llm, 
    tools=tools, 
    system_prompt=SYSTEM_PROMPT).with_config(max_iterations=3)


In [9]:
question = "Dạ cho em hỏi em là nhân viên mới đi muộn 5 phút thì bị phạt bao nhiêu tiền ạ?"

print(f"🧑 User hỏi: {question}\n")
print("🔄 Agent đang suy nghĩ và tìm kiếm...\n")
inputs = {
        "messages": [
            {"role": "user", "content": question}
        ]
    }


for chunk in agent.stream(inputs, stream_mode="updates"):
    
    node_name = list(chunk.keys())[0]
    print(f"\n[Trạm: {node_name.upper()} vừa hoàn thành]")
    
    new_message = chunk[node_name]["messages"][-1]
    
    if new_message.type == "ai":
        if new_message.tool_calls:
            print(f"🤖 AI ra quyết định: Cần gọi công cụ {new_message.tool_calls[0]['name']}")
            print(f"   Tham số truyền vào: {new_message.tool_calls[0]['args']}")
        else:
            print(f"🤖 AI trả lời user: {new_message.content[0].get('text', '')}")
            
    elif new_message.type == "tool":
        print(f"🛠️ Công cụ trả về dữ liệu (Observation):")
        # In một đoạn ngắn gọn của văn bản tìm được
        print(f"   '{new_message.content[:100]}...'")


🧑 User hỏi: Dạ cho em hỏi em là nhân viên mới đi muộn 5 phút thì bị phạt bao nhiêu tiền ạ?

🔄 Agent đang suy nghĩ và tìm kiếm...


[Trạm: MODEL vừa hoàn thành]
🤖 AI ra quyết định: Cần gọi công cụ search_fpt_policy
   Tham số truyền vào: {'query': 'quy định đi muộn phạt tiền'}

[Trạm: TOOLS vừa hoàn thành]
🛠️ Công cụ trả về dữ liệu (Observation):
   'Penalty:

* < 15 minutes: Warning
* 15 - < 30 minutes: 50,000 VND fine
* ≥ 30 minutes: Deduct 0.5 le...'

[Trạm: MODEL vừa hoàn thành]
🤖 AI trả lời user: Chào bạn, theo quy định của FPT, nếu bạn đi muộn dưới 15 phút thì sẽ nhận cảnh cáo.
